"""
=============================================================================
Tendo Model Monitoring — Monthly Power BI Data Pipeline
=============================================================================
Run this script once per month. It pulls all raw data from BigQuery / GCS, applies every transformation from the original analysis notebook, and writes one CSV per dashboard page into the OUTPUT_DIR folder.

Power BI setup (one-time):
  1. In Power BI Desktop → Get Data → Text/CSV → point to each file in OUTPUT_DIR
  2. Each CSV becomes one table — one table per dashboard page
  3. Refresh monthly: run this script, then hit Refresh in Power BI
  
Output files (one per slide/page): <br><br>
    page02_model_performance_gini_cindex.csv <br>
    page03_oop_rank_order_new_users.csv <br>
    page04_oop_rank_order_existing_users.csv <br>
    page05_attrition_rank_order_new_users.csv <br>
    page06_attrition_rank_order_existing_users.csv <br>
    page07_unit_bad_rate_mom.csv <br>
    page08_peso_bad_rate_mom.csv <br>
    page09_cl_plus_exposure.csv <br>
    page10_new_borrower_exposure.csv <br>
    page11_utilization_trend.csv <br>
=============================================================================
"""

# Import Packages

In [13]:
import os
import pickle
import warnings
from datetime import datetime, date

import numpy as np
import pandas as pd
from google.cloud import bigquery, storage
from sklearn.metrics import roc_auc_score
import re

warnings.filterwarnings("ignore")

pd.options.display.max_columns = None


# =============================================================================
# CONFIGURATION — edit these each month if needed
# =============================================================================

In [26]:
# GCP credentials — use ADC JSON or set GOOGLE_APPLICATION_CREDENTIALS env var
CREDENTIALS_PATH = r"C:\Users\Dwaipayan\AppData\Roaming\gcloud\legacy_credentials\dchakroborti@tonikbank.com\adc.json"
os.environ["GOOGLE_APPLICATION_CREDENTIALS"] = CREDENTIALS_PATH

os.environ["GOOGLE_CLOUD_PROJECT"] = "prj-prod-dataplatform"

PROJECT_ID = "prj-prod-dataplatform"
GS_BUCKET = "prod-asia-southeast1-tonik-aiml-workspace"

CLOUDPATH = "DC/Tendo_Model_Monitoring_Data"
CURRENT_DATE = datetime.now().strftime("%Y%m%d")

# Output folder — Power BI reads CSVs from here
OUTPUT_DIR = r"D:\OneDrive - Tonik Financial Pte Ltd\MyStuff\Data Engineering\Tendo_Model_Monitoring_Dashboards_Requirement\Tendo_Monitoring_data"


In [3]:
# Rolling window: script always covers Jun 2025 → current month
# Extend this dict each month by adding the new month entry
REPORT_PERIODS = {
    "Jun 2025": {"start": "2025-06-01", "end": "2025-06-30"},
    "Jul 2025": {"start": "2025-07-01", "end": "2025-07-31"},
    "Aug 2025": {"start": "2025-08-01", "end": "2025-08-31"},
    "Sep 2025": {"start": "2025-09-01", "end": "2025-09-30"},
    "Oct 2025": {"start": "2025-10-01", "end": "2025-10-31"},
    "Nov 2025": {"start": "2025-11-01", "end": "2025-11-30"},
    "Dec 2025": {"start": "2025-12-01", "end": "2025-12-31"},
    # ← ADD NEXT MONTH HERE e.g. "Jan 2026": {"start": "2026-01-01", "end": "2026-01-31"}
}


In [4]:
# Attrition segment ordering (used for sort column)
ATTRITION_ORDER = {"Very low": 1, "Low": 2, "Average": 3, "High": 4, "Very high": 5}

# OOP segment ordering
OOP_SEGMENT_ORDER = {"A": 1, "B": 2, "C": 3, "D": 4, "E": 5, "F": 6}

# =============================================================================
# HELPERS
# =============================================================================

In [23]:
def log(msg: str):
    print(f"[{datetime.now().strftime('%H:%M:%S')}] {msg}")


def save_csv(df: pd.DataFrame, filename: str):
    os.makedirs(OUTPUT_DIR, exist_ok=True)
    path = os.path.join(OUTPUT_DIR, filename)
    df.to_csv(path, index=False)
    log(f"  ✓ Saved {filename}  ({len(df):,} rows)")


def generate_bucket_url(filename, bucket_name):
    return f"gs://{bucket_name}/{filename}"


def save_df_to_gcs(df, bucket_name, destination_blob_name, file_format='csv'):
    """Saves a pandas DataFrame to Google Cloud Storage.

    Args:
        df: The pandas DataFrame to save.
        bucket_name: The name of the GCS bucket.
        destination_blob_name: The name of the blob to be created.
        file_format: The file format to save the DataFrame in ('csv' or 'parquet').
    """

    # Create a temporary file
    if file_format == 'csv':
        temp_file = 'temp.csv'
        df.to_csv(temp_file, index=False)
    elif file_format == 'parquet':
        temp_file = 'temp.parquet'
        df.to_parquet(temp_file, index=False)
    else:
        raise ValueError("Invalid file format. Please choose 'csv' or 'parquet'.")

    # Upload the file to GCS
    storage_client = storage.Client(project="prj-prod-dataplatform")

    storage_client = storage.Client()
    bucket = storage_client.bucket(bucket_name)
    blob = bucket.blob(destination_blob_name)

    blob.upload_from_filename(temp_file)

    # Remove the temporary file
    import os
    os.remove(temp_file)

import pickle
import io
from google.cloud import storage
def save_pickle_to_gcs(data, bucket_name, destination_blob_name):
    """
    Save any Python object as a pickle file to Google Cloud Storage
    
    Args:
        data: The Python object to pickle (DataFrame, dict, list, etc.)
        bucket_name: Name of the GCS bucket
        destination_blob_name: Path/filename in the bucket
    """
    # Initialize the GCS client
    client = storage.Client()
    bucket = client.bucket(bucket_name)
    blob = bucket.blob(destination_blob_name)
    
    # Serialize the data to pickle format in memory
    pickle_buffer = io.BytesIO()
    pickle.dump(data, pickle_buffer)
    pickle_buffer.seek(0)
    
    # Upload the pickle data to GCS
    blob.upload_from_file(pickle_buffer, content_type='application/octet-stream')
    print(f"Pickle file uploaded to gs://{bucket_name}/{destination_blob_name}")


def calculate_gini(data, date_col, score_col, target_col, periods, weight_col=None):
    """
    Compute Gini for each period in `periods`.
    Returns a tidy DataFrame with one row per period.
    """
    dt = data[data[target_col].notna()].copy()
    dt[date_col] = pd.to_datetime(dt[date_col]).dt.date
    rows = []

    for period_name, period_info in periods.items():
        start = pd.to_datetime(period_info["start"]).date()
        end = pd.to_datetime(period_info["end"]).date()
        mask = (dt[date_col] >= start) & (dt[date_col] <= end)
        sub = dt[mask].copy()

        sample_size = sub["ee_customer_id"].nunique()
        if sample_size == 0:
            rows.append(
                {
                    "Period": period_name,
                    "Start_Date": start,
                    "End_Date": end,
                    "Sample_Size": 0,
                    "Bad_Rate_Pct": None,
                    "Gini": None,
                }
            )
            continue

        bad_rate = 100 * (
            1
            - sub[["ee_customer_id", target_col]].drop_duplicates()[target_col].sum()
            / sample_size
        )

        if sub[target_col].nunique() < 2:
            gini = None
        else:
            try:
                kwargs = {"sample_weight": sub[weight_col]} if weight_col else {}
                auc = roc_auc_score(sub[target_col], sub[score_col], **kwargs)
                gini = round(2 * auc - 1, 4)
            except Exception:
                gini = None

        rows.append(
            {
                "Period": period_name,
                "Start_Date": start,
                "End_Date": end,
                "Sample_Size": sample_size,
                "Bad_Rate_Pct": round(bad_rate, 2),
                "Gini": gini,
            }
        )
    return pd.DataFrame(rows)

# =============================================================================
# STEP 1 — LOAD RAW DATA FROM BIGQUERY & GCS
# =============================================================================

## ==============================================================================
## Create Data for Onboarding of user 
## ==============================================================================

In [6]:
client = bigquery.Client(PROJECT_ID)

In [8]:
dt = client.query("""select * from `prj-prod-dataplatform.worktable_data_analysis.tendo_scorecard_features_data_23-03-2026`;""").to_dataframe()
print(f"The shape of the tendo scorecard feature data is: {dt.shape}")

dt_repayments = client.query("""select * from `prj-prod-dataplatform.worktable_data_analysis.tendo_scorecard_loan_repayment_data_23-03-2026`;""").to_dataframe()
print(f"The shape of the tendo scorecard repayment data is: {dt_repayments.shape}")




The shape of the tendo scorecard feature data is: (4168379, 100)
The shape of the tendo scorecard repayment data is: (16086906, 15)


## Resignation code

In [14]:
def to_date_str(val) -> str | None:
    # missing
    if val is None or pd.isna(val):
        return None

    # already a string
    if isinstance(val, str):
        return val.strip()

    # datetime-like (Timestamp, datetime, date, numpy datetime64)
    if isinstance(val, (pd.Timestamp, datetime, date, np.datetime64)):
        return pd.Timestamp(val).strftime("%Y-%m-%d")

    # fallback (numbers, objects, etc.)
    return str(val).strip()

def validate_and_convert_dates(df, column_name, output_tag_col, output_date_col,
                                min_date='1990-01-01',
                                max_date='2025-10-01',
                                min_date_col=None,
                                max_date_col=None):
    """
    Validates dates and creates tags with converted dates.

    Parameters:
    -----------
    df : pd.DataFrame
        DataFrame with date column
    column_name : str
        Name of the date column to validate
    min_date : str
        Default minimum date threshold (default: '1990-01-01')
    max_date : str
        Default maximum date threshold (default: '2025-10-01')
    min_date_col : str, optional
        Column name with per-row minimum date thresholds (overrides min_date when not null)
    max_date_col : str, optional
        Column name with per-row maximum date thresholds (overrides max_date when not null)

    Tags:
    - 'valid': Already in correct format and within range
    - 'convertible': Can be fixed (missing day, wrong format) and within range
    - 'invalid': Cannot be converted or outside valid range
    - 'null': NULL/NaN/empty values

    Returns DataFrame with:
    - date_tag: validation tag
    - date_converted: standardized date in YYYY-MM-DD format or None
    """

    default_min_threshold = pd.Timestamp(min_date)
    default_max_threshold = pd.Timestamp(max_date)

    def process_date(row):
        """Process a single date value with per-row thresholds"""

        val = row[column_name]

        # Determine thresholds for this row
        # Use per-row threshold if available and not null, otherwise use default
        if min_date_col and min_date_col in df.columns and pd.notna(row[min_date_col]):
            min_threshold = pd.Timestamp(row[min_date_col])
            # Remove timezone info if present to allow comparison
            if min_threshold.tzinfo is not None:
                min_threshold = min_threshold.tz_localize(None)
        else:
            min_threshold = default_min_threshold

        if max_date_col and max_date_col in df.columns and pd.notna(row[max_date_col]):
            max_threshold = pd.Timestamp(row[max_date_col])
            # Remove timezone info if present to allow comparison
            if max_threshold.tzinfo is not None:
                max_threshold = max_threshold.tz_localize(None)
        else:
            max_threshold = default_max_threshold

        # Handle NULL values
        if pd.isna(val):
            return 'null', None

        date_str = to_date_str(val)

        # Handle empty or 'nan' strings
        if date_str == '' or date_str.lower() == 'nan':
            return 'null', None

        # Check for invalid year pattern (must start with 19 or 20)
        year_match = re.search(r'(\d{4})', date_str)
        if year_match:
            year_str = year_match.group(1)
            first_digit = year_str[0]
            second_digit = year_str[1]

            # Year must start with 1 or 2
            if first_digit not in ['1', '2']:
                return 'invalid', None

            # If starts with 1, second digit must be 9 (1900-1999)
            if first_digit == '1' and second_digit != '9':
                return 'invalid', None

            # If starts with 2, second digit must be 0 (2000-2099)
            if first_digit == '2' and second_digit != '0':
                return 'invalid', None

        # Try to parse and convert the date
        converted_date = None
        needs_conversion = False

        # Pattern 1: YYYY-MM-DD (standard format with various separators)
        for sep in ['-', '/', '.']:
            pattern = f'^(\\d{{4}}){re.escape(sep)}(\\d{{1,2}}){re.escape(sep)}(\\d{{1,2}})$'
            match = re.match(pattern, date_str)
            if match:
                year, month, day = match.groups()
                needs_conversion = False  # Reset for each pattern

                # Store original to check if padding needed
                original_month_len = len(month)
                original_day_len = len(day)

                # Pad month and day if needed
                month = month.zfill(2)
                day = day.zfill(2)

                # Check if padding was needed
                if original_month_len == 1 or original_day_len == 1:
                    needs_conversion = True

                # DON'T swap - the input format is already YYYY-MM-DD
                # (We only swap for DD-MM-YYYY format in Pattern 4)

                try:
                    parsed = pd.Timestamp(f"{year}-{month}-{day}")
                    converted_date = f"{year}-{month}-{day}"

                    # Check if within valid range
                    if parsed < min_threshold or parsed > max_threshold:
                        return 'invalid', None

                    # Different separator means needs conversion
                    if sep != '-':
                        needs_conversion = True

                    return 'convertible' if needs_conversion else 'valid', converted_date
                except:
                    return 'invalid', None

        # Pattern 2: YYYY-MM (missing day)
        for sep in ['-', '/', '.']:
            pattern = f'^(\\d{{4}}){re.escape(sep)}(\\d{{1,2}})$'
            match = re.match(pattern, date_str)
            if match:
                year, month = match.groups()
                month = month.zfill(2)
                day = '01'  # Default to first day of month

                try:
                    parsed = pd.Timestamp(f"{year}-{month}-{day}")
                    converted_date = f"{year}-{month}-{day}"

                    # Check if within valid range
                    if parsed < min_threshold or parsed > max_threshold:
                        return 'invalid', None

                    return 'convertible', converted_date
                except:
                    return 'invalid', None

        # Pattern 3: YYYYMMDD (no separator)
        if re.match(r'^\d{8}$', date_str):
            year = date_str[0:4]
            month = date_str[4:6]
            day = date_str[6:8]

            try:
                parsed = pd.Timestamp(f"{year}-{month}-{day}")
                converted_date = f"{year}-{month}-{day}"

                # Check if within valid range
                if parsed < min_threshold or parsed > max_threshold:
                    return 'invalid', None

                return 'convertible', converted_date
            except:
                return 'invalid', None

        # Pattern 4: DD-MM-YYYY or MM-DD-YYYY (need to detect which)
        for sep in ['-', '/', '.']:
            pattern = f'^(\\d{{1,2}}){re.escape(sep)}(\\d{{1,2}}){re.escape(sep)}(\\d{{4}})$'
            match = re.match(pattern, date_str)
            if match:
                part1, part2, year = match.groups()
                part1 = part1.zfill(2)
                part2 = part2.zfill(2)

                # Determine if DD-MM-YYYY or MM-DD-YYYY
                # If part1 > 12, it must be day, so DD-MM-YYYY
                if int(part1) > 12:
                    day, month = part1, part2
                # If part2 > 12, it must be day, so MM-DD-YYYY
                elif int(part2) > 12:
                    month, day = part1, part2
                # Ambiguous case: assume MM-DD-YYYY (common US format)
                else:
                    # Could be either, default to MM-DD-YYYY but mark as needing review
                    month, day = part1, part2

                try:
                    parsed = pd.Timestamp(f"{year}-{month}-{day}")
                    converted_date = f"{year}-{month}-{day}"

                    # Check if within valid range
                    if parsed < min_threshold or parsed > max_threshold:
                        return 'invalid', None

                    return 'convertible', converted_date
                except:
                    return 'invalid', None

        # If nothing matched, it's invalid
        return 'invalid', None

    # Apply processing to all rows
    results = df.apply(process_date, axis=1)

    df_result = pd.DataFrame(index=df.index)
    df_result[output_tag_col] = results.apply(lambda x: x[0])
    df_result[output_date_col] = pd.to_datetime(results.apply(lambda x: x[1]))

    return df_result

# loan table preparation
str_columns = ['ee_customer_id', 'ln_loan_id'] 

missing_values = ['<NA>', 'None', 'NaN', 'nan', '', 'null', 'NULL']
dt[str_columns] = dt[str_columns].replace(missing_values, np.nan).astype('string')

localize_date_cols = ['ee_permanent_freeze_date','ee_onboarding_date']
for col in localize_date_cols:
    print(col)
    dt[col] = pd.to_datetime(dt[col]).dt.tz_localize(None)

date_col_format = ['ee_resignation_date']
for date_col in date_col_format:
    dt[date_col] = pd.to_datetime(dt[date_col], format='%Y-%m-%d', errors='coerce')

# repayment table preparation
dt_repayments = dt_repayments.sort_values(
    ['loan_id', 'repaid_date', 'installment_number', 'outstanding_balance', 'paid_amount'],
    ascending=[True, True, True, False, False]
)

dt_repayments['repaid_date'] = pd.to_datetime(dt_repayments['repaid_date'], format='%Y-%m-%d')
dt_repayments['due_date'] = pd.to_datetime(dt_repayments['due_date'], format='%Y-%m-%d')

dt_repayments['loan_id'] = dt_repayments['loan_id'].astype('str')
dt_repayments['user_id'] = dt_repayments['user_id'].astype('str')



resignation_dates = validate_and_convert_dates(
    df=dt.groupby('ee_customer_id')[['ee_resignation_date']].first(),
    column_name='ee_resignation_date',
    output_tag_col='ee_resignation_date_tag',
    output_date_col='ee_resignation_date_validated',
    min_date=dt['ee_onboarding_date'].min().strftime(format='%Y-%m-%d'),
    max_date='2026-03-23',
    min_date_col=None,
    max_date_col=None
)

permanent_freeze_dates = validate_and_convert_dates(
    df=dt.groupby('ee_customer_id')[['ee_permanent_freeze_date']].first(),
    column_name='ee_permanent_freeze_date',
    output_tag_col='ee_permanent_freeze_date_tag',
    output_date_col='ee_permanent_freeze_date_validated',
    min_date=dt['ee_onboarding_date'].min().strftime(format='%Y-%m-%d'),
    max_date='2026-03-23',
    min_date_col=None,
    max_date_col=None
)

resignation_dates = resignation_dates.merge(
        permanent_freeze_dates['ee_permanent_freeze_date_validated'],
        how='left',
        on='ee_customer_id'
)

resignation_dates['ee_resignation_date_correct'] = resignation_dates.loc[:, 'ee_resignation_date_validated'].fillna(resignation_dates['ee_permanent_freeze_date_validated'])

dt = dt.merge(
    resignation_dates['ee_resignation_date_correct'].reset_index(),
    how='left',on='ee_customer_id'
)

dt_repayments['vendor_id_shifted'] = dt_repayments.groupby('loan_id')[['vendor_id']].shift(-1)
dt_repayments['repaid_date_shifted'] = dt_repayments.groupby('loan_id')[['repaid_date']].shift(-1)

resignation_dates_repayments = (
    dt_repayments[['user_id','loan_id','vendor_id','vendor_id_shifted','repaid_date']]
    .query('vendor_id == "TPSD"')
    .fillna('TPSD')
    .query('vendor_id != vendor_id_shifted & vendor_id_shifted == "TPFP"')
    .assign(resignation_date_fp_new = lambda x: x['repaid_date'] + pd.DateOffset(months=1))
    .groupby('user_id')[['resignation_date_fp_new']]
    .max()
    .reset_index()
)

dt = dt.merge(resignation_dates_repayments, how='left', left_on='ee_customer_id', right_on='user_id')
dt['ee_resignation_date_correct'] = dt['ee_resignation_date_correct'].fillna(dt['resignation_date_fp_new'])

ee_permanent_freeze_date
ee_onboarding_date


In [15]:
dt.head()

,ee_customer_id,ee_email,ee_phone_number,ee_firstname,ee_middlename,ee_lastname,ee_birthdate,ee_gender,ee_email_verified_flag,ee_telephone_verified_flag,ee_id_verified_flag,ee_income_verified_flag,ee_morning_time_contact_time,ee_afternoon_time_contact_time,ee_account_activated_flag,ee_region_name,ee_city_name,ee_barangay,ee_address_line_1,ee_address_line_2,ee_postal_code,ee_landmark,ee_residing_date,ee_employment_status,ee_civil_status,ee_kyc_doc_name,ee_is_citizen_flag,ee_onboarding_date,ee_employment_date,ee_fraud_status,ee_job_type,ee_comment,ee_department,ee_recommended_ir,ee_job_title,ee_employment_type,ee_nature_of_work,ee_permanent_freeze_date,ee_resignation_date,ee_product_type,ee_frozen_status,cust_risk_employer_time_score_v1,cust_risk_gender_score_v1,cust_risk_declared_income_score_v1,cust_risk_civil_status_score_v1,cust_risk_combined_score_v1,cust_risk_cat_v1,er_employer_id,er_employer_group_id,er_employer_name,er_email_domain,er_repayment_days_month,er_custom_email,er_payment_reminders,er_address,er_postal_code_id,er_max_base_interest_rate,er_employer_industry,er_employer_status,er_activated_at,er_deleted_at,er_created_at,er_updated_at,cl_monthly_utility_bills_amount,cl_monthly_income_gross,cl_monthly_income_net,cl_multiple_purchases_enabled_flag,cl_max_credit_limit_multiplier,cl_max_debt_income_ratio,cl_matured_fpd30_flag,cl_matured_fspd30_flag,cl_matured_fstpd30_flag,cl_matured_fstfpd30_flag,cl_fpd30_flag,cl_fspd30_flag,cl_fstpd30_flag,cl_fstfpd30_flag,ln_loan_id,ln_loan_type,ln_loan_status,ln_disbursement_channel,ln_original_principal,ln_orig_interest_fees,ln_orig_tenor,ln_loan_application_datetime,ln_repaid_full_flag,ln_fully_repaid_date,ln_fpd30_flag,ln_fspd30_flag,ln_fstpd30_flag,ln_fstfpd30_flag,ln_min_loan_due_date,ln_os_principal_at_fpd30,ln_os_principal_at_fspd30,ln_os_principal_at_fstpd30,ln_os_principal_at_fstfpd30,ln_matured_fpd30_flag,ln_matured_fspd30_flag,ln_matured_fstpd30_flag,ln_matured_fstfpd30_flag,ee_resignation_date_correct,user_id,resignation_date_fp_new
0,1166660,edwardjay1016@gmail.com,+63 (956) 746 1133,Cleo Fatima,Villanueva,Eguia,1988-02-09,Female,1,1,2,2,None,None,2,NCR - NATIONAL CAPITAL REGION,None,SANTA LUCIA,8 Lobeban Street Soldiers Village,None,None,None,None,regular,Single,Barangay Certificate w/ picture,true,2024-04-22 16:28:58,None,Normal,None,None,None,None,LEAD CONSULTANT,Permanent Job (Private sector),None,NaT,NaT,employer,Frozen,183,120,146,117,566,A,10225.000000000,None,CGI PHILIPPINES INC.,None,"[""5"",""20""]",1,-1,None,None,3.5,9,IN_PROGRESS,2024-03-19 13:48:32.000000,NaT,2024-03-19 13:48:32+00:00,2026-01-09 11:59:31+00:00,6000,68500,55000,1,188,40,1,1,1,1,1,1,1,1,4875492,Tendo,Approved/Disbursed,PH_BPI,103400.0,28952.0,10,2025-11-11 14:37:19+00:00,0,None,1,1,1,1,2025-12-05,99342.277,0.000,0.0,0.0,1,1,1,0,NaT,NaN,NaT
1,778469,daisyvegino@icloud.com,+63 (916) 940 1823,Daisy,Gamos,Vegino,1998-03-22,Female,1,1,2,2,None,None,2,Region III,None,DAU,1931 Aquino st. doña Maria Subd.,None,None,MEGASAVERS,None,regular,Single,None,true,2022-05-15 10:26:46,None,Risk,None,"110,Resigned",None,None,Credit and Collection Specialist,Permanent Job (Private sector),None,2022-07-29 06:19:13,2022-07-26,employer,Frozen,123,120,118,117,478,D,10106.000000000,None,Origo BPO - PHILS Limited,None,"[""15"",""30""]",1,14,"B24, Manunggal Street, Berthaphil Park II, Jos...",510,4.0,8,IN_PROGRESS,2022-01-21 22:27:46.000000,NaT,2022-01-21 22:27:46+00:00,2025-08-28 01:35:19+00:00,3000,28000,24000,1,240,30,1,1,1,1,0,1,1,1,399454,Tendo,Approved/Disbursed,None,6300.0,1008.0,4,2022-06-24 12:04:35+00:00,0,None,0,1,1,1,2022-07-30,0.000,4879.384,0.0,0.0,1,1,1,1,2022-07-26,NaN,NaT
2,1133408,amheerjohnt@gmail.com,+63 (965) 855 8790,Jackielen,Faca,Fabillo,1985-06-06,Female,1,1,2,2,None,None,2,None,None,None,L20 B7 P2 Lakeside Nest Subdivision,None,None,None,None,probationary,Single,Payslip,true,2024-01-09 13:47:31,2023-03-16,Risk,None,None,None,None,Merchandiser,Permanent Job (Private sector),None,NaT,NaT,employ

In [27]:
# Your modified code
bucket_name = GS_BUCKET
pickle_filename = f"resignation_data_{CURRENT_DATE}"

# Construct the new filename with .pkl extension
data_filename = f"{pickle_filename}.pkl"
print(data_filename)

destination_blob_name = f"{CLOUDPATH}/{data_filename}"

# Save the DataFrame (or any object) as pickle to GCS
save_pickle_to_gcs(dt[['ee_customer_id','ee_resignation_date_correct']], bucket_name, destination_blob_name)



resignation_data_20260324.pkl
Pickle file uploaded to gs://prod-asia-southeast1-tonik-aiml-workspace/DC/Tendo_Model_Monitoring_Data/resignation_data_20260324.pkl


## Outstanding balance code

In [16]:
dt_repayments = dt_repayments.sort_values(
    ['loan_id', 'repaid_date', 'installment_number', 'outstanding_balance', 'paid_amount'],
    ascending=[True, True, True, False, False]
)

# converting loan_id and user_id to str in order to be aligned with loan data
dt_repayments['loan_id'] = dt_repayments['loan_id'].astype('str')
dt_repayments['user_id'] = dt_repayments['user_id'].astype('str')

# merging with loan data to get columns for actual osbal calculations
dt_repayments = dt_repayments.merge(dt[['ln_loan_id','ln_original_principal', 'ln_orig_interest_fees']].drop_duplicates(),
                                    how='left', left_on='loan_id', right_on='ln_loan_id')

# converting dates
dt_repayments['repaid_date'] = pd.to_datetime(dt_repayments['repaid_date'], format='%Y-%m-%d')
dt_repayments['due_date'] = pd.to_datetime(dt_repayments['due_date'], format='%Y-%m-%d')

# Calculate cumulative paid amount per loan
dt_repayments['cumulative_paid_principal'] = dt_repayments.groupby('loan_id')['paid_principal'].cumsum()

# Calculate osbal_after directly from loan_amount - cumulative_paid
dt_repayments['osbal_after'] = dt_repayments['ln_original_principal'] - dt_repayments['cumulative_paid_principal']

# Calculate osbal_before by shifting osbal_after and using loan_amount for first payment
dt_repayments['osbal_before'] = dt_repayments.groupby('loan_id')['osbal_after'].shift(1)

# Fill first payment osbal_before with loan_amount
mask = dt_repayments['osbal_before'].isna()
dt_repayments.loc[mask, 'osbal_before'] = dt_repayments.loc[mask, 'ln_original_principal']

In [17]:
dt_repayments.sample(2)

,user_id,loan_id,installment_number,vendor_id,subfamily,repaid_date,due_date,due_principal,due_interest,due_amount,paid_principal,paid_interest,paid_amount,outstanding_balance,DPD,vendor_id_shifted,repaid_date_shifted,ln_loan_id,ln_original_principal,ln_orig_interest_fees,cumulative_paid_principal,osbal_after,osbal_before
7985166,1374346,3281909,18,TPSD,COPA,2026-01-27,2026-01-15,636.822,319.778,956.60,636.822,319.778,956.60,14135.833,12,TPSD,2026-02-11,3281909,23750.0,10687.5,9614.167,14135.833,14772.655
149475,1117595,1043891,6,TPSD,COPA,2024-02-15,2024-02-15,71.446,36.884,108.33,71.446,36.884,108.33,1593.883,0,TPSD,2024-02-29,1043891,2000.0,600.0,406.116,1593.884,1665.330


**The tables required for this report:**

1.  `prj-prod-dataplatform.tendo_mart.tendo_scorecard_master_table_api` -   dt_prod_api
2.  `prj-prod-dataplatform.tendo_mart.tendo_scorecard_master_table` -   dt_prod_batch
3.  `prj-prod-dataplatform.tendo_mart.tendo_collection_target_master`   -  dt_oop_targets 
4.  `prj-prod-dataplatform.risk_mart.tendo_backscored_new_users_jan23_feb26_20260301_oop_with_osbal`    -   dt_bs_oop_new
5.  `prj-prod-dataplatform.risk_mart.tendo_backscored_existing_users_jan23_feb26_20260301_oop`  -   dt_bs_oop_ex
6.  `prj-prod-dataplatform.risk_mart.tendo_backscored_jan24_feb26_20260301_attrition`   -   dt_bs_attr   
7.  `prj-prod-dataplatform.worktable_data_analysis.tendo_scorecard_features_data_23-03-2026`    -   dt_bs_attr
8.  `prj-prod-dataplatform.worktable_data_analysis.tendo_scorecard_loan_repayment_data_23-03-2026`  -   dt_res

In [ ]:
log("=== STEP 1: Loading raw data ===")


# --- Production API scores (new users) ---
log("Loading prod API scores (new users)...")
dt_prod_api = client.query(
    """
    SELECT
        employee_id           AS ee_customer_id,
        run_date,
        ee_attrition_risk_segment   AS attrition_risk_segment,
        ee_attrition_time_to_leave  AS attrition_time_to_leave,
        oop_score                   AS oop_score_prod,
        oop_risk_segment            AS oop_risk_segment_prod
    FROM `prj-prod-dataplatform.tendo_mart.tendo_scorecard_master_table_api`
"""
).to_dataframe()

# --- Production batch scores (existing users) ---
log("Loading prod batch scores (existing users)...")
dt_prod_batch = client.query(
    """
    SELECT
        employee_id           AS ee_customer_id,
        run_date,
        ee_attrition_risk_segment   AS attrition_risk_segment,
        ee_attrition_time_to_leave  AS attrition_time_to_leave,
        oop_score                   AS oop_score_prod,
        oop_risk_segment            AS oop_risk_segment_prod
    FROM `prj-prod-dataplatform.tendo_mart.tendo_scorecard_master_table`
"""
).to_dataframe()

# --- OOP matured targets ---
log("Loading OOP targets...")
dt_oop_targets = client.query(
    """
    SELECT user_id AS ee_customer_id, target AS oop_target
    FROM `prj-prod-dataplatform.tendo_mart.tendo_collection_target_master`
    WHERE target_maturity_flag = 1
"""
).to_dataframe()

# --- Backscored OOP new users ---
log("Loading backscored OOP — new users...")
dt_bs_oop_new = client.query(
    """
    SELECT *
    FROM `prj-prod-dataplatform.risk_mart.tendo_backscored_new_users_jan23_feb26_20260301_oop_with_osbal`
"""
).to_dataframe()

# --- Backscored OOP existing users ---
log("Loading backscored OOP — existing users...")
dt_bs_oop_ex = client.query(
    """
    SELECT *
    FROM `prj-prod-dataplatform.risk_mart.tendo_backscored_existing_users_jan23_feb26_20260301_oop`
"""
).to_dataframe()

# --- Backscored attrition ---
log("Loading backscored attrition...")
dt_bs_attr = client.query(
    """
    SELECT *
    FROM `prj-prod-dataplatform.risk_mart.tendo_backscored_jan24_feb26_20260301_attrition`
"""
).to_dataframe()

# --- Raw features (onboarding dates etc.) from GCS ---
log("Loading raw feature data from GCS...")
dt_raw = pd.read_pickle(
    generate_bucket_url("Oleh/tendo/data/raw_data_14012026.pkl", GS_BUCKET)
)

# --- Resignation data from GCS ---
log("Loading resignation data from GCS...")
dt_res = pd.read_pickle(
    generate_bucket_url(f"{CLOUDPATH}/{data_filename}", GS_BUCKET)
)

log("All raw data loaded.")

[15:43:54] === STEP 1: Loading raw data ===
[15:43:54] Loading prod API scores (new users)...
[15:43:59] Loading prod batch scores (existing users)...
[15:44:03] Loading OOP targets...
[15:44:05] Loading backscored OOP — new users...
[15:44:07] Loading backscored OOP — existing users...
[15:44:11] Loading backscored attrition...
[15:44:16] Loading raw feature data from GCS...
[15:51:23] Loading resignation data from GCS...
[15:51:33] All raw data loaded.


In [30]:
dt_res.shape

(4168379, 2)

In [32]:
dt_raw.shape

(3812504, 100)

# =============================================================================
# STEP 2 — SHARED TRANSFORMATIONS
# =============================================================================

In [ ]:
log("=== STEP 2: Shared transformations ===")

# --- Raw features ---
dt_raw["ee_customer_id"] = dt_raw["ee_customer_id"].astype("str")
dt_raw["ee_onboarding_date"] = pd.to_datetime(
    dt_raw["ee_onboarding_date"]
).dt.tz_localize(None)

# --- Backscored OOP new ---
dt_bs_oop_new["ee_customer_id"] = dt_bs_oop_new["ee_customer_id"].astype("str")

# --- Backscored OOP existing: date offset -1 month ---
dt_bs_oop_ex["ee_customer_id"] = dt_bs_oop_ex["ee_customer_id"].astype("str")
dt_bs_oop_ex["calc_date"] = pd.to_datetime(dt_bs_oop_ex["calc_date"], errors="coerce")
dt_bs_oop_ex["calc_date_correct"] = dt_bs_oop_ex["calc_date"] - pd.DateOffset(months=1)
dt_bs_oop_ex["calc_date_ym"] = (
    dt_bs_oop_ex["calc_date_correct"].dt.year * 100
    + dt_bs_oop_ex["calc_date_correct"].dt.month
)

# --- Backscored attrition: date offset + new-customer flag ---
dt_bs_attr["ee_customer_id"] = dt_bs_attr["ee_customer_id"].astype("str")
dt_bs_attr["calc_date"] = pd.to_datetime(dt_bs_attr["calc_date"], errors="coerce")
dt_bs_attr["calc_date_correct"] = dt_bs_attr["calc_date"] - pd.DateOffset(months=1)
dt_bs_attr["calc_date_ym"] = (
    dt_bs_attr["calc_date_correct"].dt.year * 100
    + dt_bs_attr["calc_date_correct"].dt.month
)
dt_bs_attr["is_new_customer_flag_1m"] = (
    dt_bs_attr["ee_onboarding_month"] == dt_bs_attr["calc_date_ym"]
).astype(int)

# --- Prod batch: dedup + date fields ---
dt_prod_batch = dt_prod_batch.drop_duplicates(
    subset=["ee_customer_id", "run_date", "attrition_time_to_leave", "oop_score_prod"]
)
dt_prod_batch["run_date"] = pd.to_datetime(dt_prod_batch["run_date"], errors="coerce")
dt_prod_batch["run_date_ym"] = (
    dt_prod_batch["run_date"].dt.year * 100 + dt_prod_batch["run_date"].dt.month
)

# --- Prod API: date fields ---
dt_prod_api["run_date"] = pd.to_datetime(dt_prod_api["run_date"]).dt.normalize()
dt_prod_api["run_date_ym"] = (
    dt_prod_api["run_date"].dt.year * 100 + dt_prod_api["run_date"].dt.month
)

log("Shared transformations complete.")